# FinMoE — experiment grid on Kaggle

Settings: **Accelerator → GPU T4 x2**, **Internet → on**. Then *Save Version → Save & Run All (Commit)*;
the notebook runs in the background and `finmoe_results.zip` appears in the *Output* tab.

* Two job queues run in parallel, one per GPU.
* No new run is started after `DEADLINE_HOURS`, so the session ends before Kaggle's 12 h limit
  and the outputs are saved. Runs whose results are already committed in `results/runs/` are
  skipped, so after committing a partial archive the notebook can simply be run again.

In [ ]:
import os, subprocess
REPO = "https://github.com/2Gpb/Moe.git"
# private repository: add a Kaggle secret named GITHUB_TOKEN (Add-ons -> Secrets)
try:
    from kaggle_secrets import UserSecretsClient
    REPO = REPO.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")
except Exception:
    pass
if not os.path.exists("/kaggle/working/Moe"):
    subprocess.run(["git", "clone", "-q", REPO, "/kaggle/working/Moe"], check=True)
os.chdir("/kaggle/working/Moe")
!git log --oneline | head -3

In [ ]:
# keep Kaggle's CUDA build of torch, install the project and the rest with uv
!pip install -q uv
!uv pip install --system -q -e .
!python -c "import torch, transformers; print(torch.__version__, transformers.__version__, torch.cuda.device_count(), torch.cuda.get_device_name(0))"

In [ ]:
# stop here if anything is broken: the grid would otherwise fail after hours of training
subprocess.run(["python", "-m", "pytest", "-q", "tests"], check=True)

## Training grid

In [ ]:
SEEDS = ["0", "1", "2"]
DEADLINE_HOURS = "9.5"
QUEUES = {
    "0": ["esft_token", "esft_random", "router", "dense_lora_attn"],
    "1": ["esft_specific", "lora_attn", "lora_experts_r1"],
}
procs = []
for gpu, configs in QUEUES.items():
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=gpu, PYTHONUNBUFFERED="1")
    log = open(f"/kaggle/working/gpu{gpu}.log", "w")
    cmd = ["python", "scripts/run_grid.py", "--configs", *configs, "--seeds", *SEEDS, "--deadline-hours", DEADLINE_HOURS]
    procs.append(subprocess.Popen(cmd, env=env, stdout=log, stderr=subprocess.STDOUT))
print("exit codes:", [p.wait() for p in procs])

In [ ]:
!grep -h -e FAILED -e deadline -e '"run"' /kaggle/working/gpu*.log
!ls results/runs

## Frozen-model baselines and routing on the test set

In [ ]:
env = dict(os.environ, CUDA_VISIBLE_DEVICES="0")
jobs = {
    "results/eval/base_2shot_test.json": ["python", "scripts/evaluate.py", "--config", "configs/base.yaml", "--shots", "2"],
    "results/routing/base_test.pt": ["python", "scripts/analyze_routing.py", "--split", "test", "--max-examples", "0"],
}
for out, cmd in jobs.items():
    if os.path.exists(out):
        print("skip, already there:", out)
    elif subprocess.run(cmd, env=env).returncode != 0:
        print("failed:", " ".join(cmd))

In [ ]:
!zip -qr /kaggle/working/finmoe_results.zip results/runs results/eval results/routing
!ls -lh /kaggle/working/finmoe_results.zip

Download `finmoe_results.zip` from the Output tab and unzip it in the repository root
(`unzip -o finmoe_results.zip`). Then `uv run python scripts/make_tables.py` and
`uv run python scripts/make_figures.py` rebuild the tables and figures with all seeds.